## deps and data setup

In [3]:
import os
import random
import cv2
import numpy as np
from matplotlib import pyplot as plt

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Layer, Conv2D, Dense, MaxPooling2D, Input, Flatten
import tensorflow as tf

print('TensorFlow version:', tf.__version__)
print('Visible GPUs:', tf.config.list_physical_devices('GPU'))

TensorFlow version: 2.16.2
Visible GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [4]:
POS_PATH = os.path.join('../data', 'positive')
NEG_PATH = os.path.join('../data', 'negative')
ANC_PATH = os.path.join('../data', 'anchor')

os.makedirs(POS_PATH, exist_ok=True)
os.makedirs(NEG_PATH, exist_ok=True)
os.makedirs(ANC_PATH, exist_ok=True)

In [ ]:
import kagglehub, shutil, os, tarfile
from pathlib import Path
#https://www.kaggle.com/datasets/atulanandjha/lfwpeople
path = kagglehub.dataset_download("atulanandjha/lfwpeople")

# 1. Extract the tgz
with tarfile.open(f"{path}/lfw-funneled.tgz") as tar:
    tar.extractall(path)
# 2. Find all jpgs in all subfolders and copy them flatly (shallow)
for img in Path(path).rglob("*.jpg"):
    shutil.copy2(img, f"{NEG_PATH}/{img.name}")

print("Data downloaded!")

In [5]:

import uuid

window_name = 'frame'
cap = cv2.VideoCapture(0)
frame = None

try:
    if not cap.isOpened():
        raise RuntimeError('Could not open webcam')

    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    while True:
        ret, frame = cap.read()
        if not ret or frame is None:
            print('Webcam read failed, stopping capture loop.')
            break

        #250x250px center crop
        frame = frame[120:120+250,200:200+250, :]

         # Collect anchors
        if cv2.waitKey(1) & 0XFF == ord('a'):
            imgname = os.path.join(ANC_PATH, '{}.jpg'.format(uuid.uuid1()))
            cv2.imwrite(imgname, frame)

        # Collect positives
        if cv2.waitKey(1) & 0XFF == ord('p'):
            imgname = os.path.join(POS_PATH, '{}.jpg'.format(uuid.uuid1()))
            cv2.imwrite(imgname, frame)

        cv2.imshow(window_name, frame)

        # Exit on q, ESC, or close of OpenCV window.
        if cv2.waitKey(1) & 0XFF == ord('q') or cv2.waitKey(1) == 27:
            break

        if cv2.getWindowProperty(window_name, cv2.WND_PROP_VISIBLE) < 1:
            break
except KeyboardInterrupt:
    print('Capture interrupted by user.')
finally:
    cap.release()
    cv2.destroyAllWindows()
    cv2.waitKey(1)


## Data Preprocessing